In [ ]:
import requests
import pandas as pd
import time
from tqdm import tqdm
from google.colab import files

##############################################################
# USER SETTINGS
##############################################################

# Your Open Access API Key
SPRINGER_API_KEY = "808b602a8e88ede55d5e4ae1f1d35eee"

# Open Access Endpoint
url = "https://api.springernature.com/openaccess/json"

# Clean search string without ANY boolean keywords (AND, OR) or parentheses ()
SPRINGER_QUERY = "bee pesticide transcriptomics expression omics"

params = {
    "q": SPRINGER_QUERY,
    "api_key": SPRINGER_API_KEY,
    "p": 100,  # Max records per batch
    "s": 1
}

print(f"Connecting to Springer Open Access API using query: '{SPRINGER_QUERY}'...")
r = requests.get(url, params=params)

if r.status_code != 200:
    print(f"\n❌ API Error ({r.status_code}): {r.text}")
    r.raise_for_status()

data = r.json()
total_count = int(data.get("result", [{}])[0].get("total", 0))

print(f"Connected successfully!")
print(f"Total Springer Open Access records found: {total_count}\n")

##############################################################
# FETCH ALL PAPERS
##############################################################

all_records = []
batch_size = 100
max_records = min(total_count, 3000)

for start in tqdm(range(1, max_records, batch_size), desc="Downloading Papers"):
    params["s"] = start
    resp = requests.get(url, params=params)

    if resp.status_code == 200:
        items = resp.json().get("records", [])

        for item in items:
            pub_date = item.get("publicationDate", "")
            pub_year = int(pub_date[:4]) if len(pub_date) >= 4 and pub_date[:4].isdigit() else 0

            # Enforce 2010–2026 publication window
            if 2010 <= pub_year <= 2026:
                creators = item.get("creators", [])
                author_names = [c.get("creator", "") for c in creators if "creator" in c]

                url_list = item.get("url", [])
                web_url = url_list[0].get("value", "") if url_list else ""

                all_records.append({
                    "PMID": "",
                    "DOI": item.get("doi", ""),
                    "Title": item.get("title", ""),
                    "Authors": "; ".join(author_names),
                    "Journal": item.get("publicationName", ""),
                    "Year": pub_year,
                    "Abstract": item.get("abstract", ""),
                    "Springer_URL": web_url
                })
    time.sleep(0.3)

##############################################################
# EXPORT
##############################################################

df_springer = pd.DataFrame(all_records)

filename = "Springer_OpenAccess_Bee_Pesticide_Omics_2010_2026.xlsx"
df_springer.to_excel(filename, index=False, engine='openpyxl')

print(f"\nExtraction Complete!")
print(f"Total Open Access records downloaded: {len(df_springer)}")
print(df_springer.head(3))

# Auto-download file in Google Colab
files.download(filename)

Connecting to Springer Open Access API using query: 'bee pesticide transcriptomics expression omics'...

❌ API Error (403): {"status":"Fail","message":"Access to this resource is restricted. This is a premium feature","error":{"error":"Forbidden","error_description":"You are attempting to access a premium feature. Please write to supportapi@springernature.com for assistance."}}


HTTPError: 403 Client Error: Forbidden for url: https://api.springernature.com/openaccess/json?q=bee+pesticide+transcriptomics+expression+omics&api_key=808b602a8e88ede55d5e4ae1f1d35eee&p=100&s=1